# 1. Vector Search - Embedding

First of all we're going to see how different documents are embedded into vectors. This is important so that we can find the closest semantic vector to the question from the user, which will allow us to understand which of the documents provides (most likely) the answer to the question.

## 1.1 Vector Search - Theory and Manual Approach (numpy)

In [ ]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [ ]:
from _ingest import load_data

documents = load_data()
texts = []

for doc in documents:
    text = doc["question"] + " " + doc["answer"]
    texts.append(text)

In [4]:
from tqdm.auto import tqdm

batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)


  0%|          | 0/27 [00:00<?, ?it/s]

In [5]:
import numpy as np
X = np.array(vectors)

In [ ]:
# We'll now take the question we want to ask and see which of the documents is closer semantically to it.

# We're first of all encoding the question in the same way that we encoded the documents in the knowledge base.
query = "Can I still join the course after it has begun?"
v_query = model.encode(query)

# We now take the matrix-vector multiplication. We can check vector by vector the cosine similarity to the question and find out the best scores (matches).
scores = X.dot(v_query)

In [ ]:
# To find the top match we can use the np.argmax function.

idx = np.argmax(scores)
idx, scores[idx]

(np.int64(538), np.float32(0.74807596))

In [9]:
# To obtain the top 5 results, for example, we can use the np.argsort function.

top5 = np.argsort(scores)[-5:][::-1]
top5

array([538, 907, 625,   2,   7])

In [10]:
scores[top5]

array([0.74807596, 0.73913485, 0.73142934, 0.72020006, 0.64623964],
      dtype=float32)

In [ ]:
# We can see how all 5 top scored documents have similar meanings than the asked question. The LLM would then use them to formulate a response.

for idx in top5:
    print(scores[idx])
    print(documents[idx])
    print()

0.74807596
{'id': '74eb249bbf', 'course': 'llm-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'I just discovered the course. Can I still join?', 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}

0.73913485
{'id': '41aabbd7c5', 'course': 'machine-learning-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'The course has already started. Can I still join it?', 'answer': 'Yes, you can. Even though you missed the start date, you can register for the course. You won’t be able to submit some of the homeworks, but you can still take part in the course.\n\nIn order to get a certificate, you need to submit 2 out of 3 course projects and review 3 peers by the deadline. It means that if you join the course at the end of November and manage to work on two projects, you will still be eligible for a certificate.'}

0.73142934
{'id': '2d8b16c2a0', 'course': 'mlops-zoomcamp', 

## 1.2 Vector Search (minsearch)

In [12]:
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=["course"])
vindex.fit(X, documents)

In [ ]:
# This approach takes all 1200 documents and selects the top 5. 

query = "I just discovered the course. Can I still join it?"
query_vector = model.encode(query)

results = vindex.search(query_vector, num_results=5)

In [14]:
results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '41aabbd7c5',
  'course': 'machine-learning-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'The course has already started. Can I still join it?',
  'answer': 'Yes, you can. Even though you missed the start date, you can register for the course. You won’t be able to submit some of the homeworks, but you can still take part in the course.\n\nIn order to get a certificate, you need to submit 2 out of 3 course projects and review 3 peers by the deadline. It means that if you join the course at the end of November and manage to work on two projects, you will still be eligible for a certificate.'},
 {'id': '2d8b16c2a0',
  'course': 'mlops-zoomcamp',
  'section':

In [15]:
# As we saw in the previous lesson, we can often filter the search by including some keywords, in this case, we can use the course.

results_llm = vindex.search(
    query_vector,
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

results_llm

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you can only get a certificate if you finish the course with a "live" cohort.\n\nWe don\'t award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\n\nYou can only peer-review projects at the time the course is running; after the form is closed and the peer-review list is compiled.'},
 {'id': 'bd31146b0e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'When will the co

## 1.3 Vector Search with RAG

In [16]:
from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
openai_client = OpenAI()

In [17]:
from _ingest import load_data, build_index

documents = load_data()
index = build_index(documents)

We will be re-using the RAGBase class, but we'll adapt it for the current goal. 
We cannot pass some arguments as the class currently exists, so we will be creating a new class, RAGVector.
This is called inheriting, and we're able to take all existing functions from the existing class, while we can also override functions with new definitions.

In [23]:
from _rag_helper import RAGBase

class RAGVector(RAGBase):

# We add an extra argument, embedder, using the __init__ method, which will be used in the sentence transformer.

    def __init__(self, embedder, **kwargs):
        super().__init__(**kwargs)
        self.embedder = embedder

# In search, we override the existing function with a new one, that turns the query into a vector.

    def search(self, query):
        query_vector = self.embedder.encode(query)
        filter_dict = {"course": self.course}

        return self.index.search(
            query_vector,
            num_results = self.matches,
            filter_dict = filter_dict
        )

In [24]:
# We will use the same model and index we used in the previous sections of this lecture.

vector_assistant = RAGVector(
    embedder = model,
    index = vindex,
    llm_client = openai_client
)

In [29]:
user_qn = "The course has started already, am I still able to signup?"

vector_assistant.rag(user_qn)

Usage: Input: 364. Output: 30.



'Yes, you can still join. If you want a certificate, you need to submit your project while submissions are still being accepted.'

## 1.4 Vector search with sqlitesearch

minsearch has some issues such as rebuilding the index every time, keeping all the vectors in memory, and searching by "brute force".
sqlitesearch helps us with these, since it allows to store the vectors in a database that we can check everytime without rebuilding the index, and it uses less space in memory.
Moreover, we currently did the Nearest Neighbour approach, meaning we calculated all vectors and found the nearest one. We can instead use Approximate Nearest Neighbour which will take the area that is most likely to have the neighbours and then it computes the nearest neighbours within the area, which means it's less resource intensive and faster.

In [26]:
from sqlitesearch import VectorSearchIndex

vs_index = VectorSearchIndex(
    keyword_fields = ["course"],
    mode = "ivf",
    db_path = "faq_vectors.db"
)

In [ ]:
# This command will now save the index to faq_vectors.db

vs_index.fit(vectors, documents)

In [ ]:
# We can also filter by course as we did before.

vs_vector = model.encode(user_qn)
results = vs_index.search(
    vs_vector,
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)
results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': 'bd31146b0e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'When will the course be offered next?',
  'answer': 'Summer 2027.'},
 {'id': 'aa310de435',
  'course': 'l

In [32]:
# If we're done with the index, then we should close it.

vs_index.close()

In [ ]:
# We can reopen the index without re-computing everything, just running:

from sentence_transformers import SentenceTransformer
from sqlitesearch import VectorSearchIndex

model = SentenceTransformer("all-MiniLM-L6-v2")

vs_index = VectorSearchIndex(
    keyword_fields=["course"],
    mode="ivf",
    db_path="faq_vectors.db"
)

vs_vector = model.encode("How do I run Kafka?")
results = vs_index.search(vs_vector, num_results=5)